# 09 - Debugging Gradients

这一节专门练梯度调试。

这一节只抓一句话：

> 训练不动时，先看四件事：grad 有没有清零、learning_rate 是否合适、ReLU 有没有断掉、loss 和 update 顺序是否正确。

这节不是学新数学，是学排错手感。

<!-- xiao-preview:start -->
## 课前预习作业：重复 backward 会发生什么

micrograd 的梯度会用 `+=` 累加。先猜一下同一张图重复 backward 的结果。

In [ ]:
# TODO: 把 None 改成你的答案。
# w = Value(3.0), L = w ** 2
# 第一次 L.backward() 后 w.grad 是多少？第二次不清零再 L.backward() 后是多少？
preview_first_grad = None
preview_second_grad = None


def qa_check_09_preview(first_grad, second_grad):
    if first_grad is None or second_grad is None:
        print('请先填写 TODO：先猜重复 backward 后的梯度。')
        return
    assert first_grad == 6, f'第一次应该是 6，但你填的是 {first_grad}'
    assert second_grad == 12, f'第二次会累加到 12，但你填的是 {second_grad}'
    print('OK: 重复 backward 预习通过。')


qa_check_09_preview(preview_first_grad, preview_second_grad)

<details>
<summary><strong>Show / Hide 提示</strong></summary>

`L = w^2` 在 `w=3` 时导数是 `6`。第二次如果没有清零，会在已有的 `6` 上继续加。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_first_grad = 6
preview_second_grad = 12
```

</details>

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import random

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))

from micrograd.engine import Value
from micrograd.nn import MLP

random.seed(1337)
print('project root:', PROJECT_ROOT)

## 1. Debug 地图

| 现象 | 先检查 |
|---|---|
| 第二次运行 cell，grad 变成两倍 | 是否重复 backward 没清零 |
| loss 不下降 | update 方向、learning_rate、grad 是否为 0 |
| loss 震荡或变大 | learning_rate 可能太大 |
| 某些参数 grad 一直是 0 | ReLU 可能在负半轴断掉 |
| 训练越来越怪 | 是否每轮都 `zero_grad -> backward -> update` |

## 2. Bug 1：重复 backward 没清零

In [ ]:
w = Value(3.0)
L = w ** 2

L.backward()
print('after first backward :', w.grad)

L.backward()
print('after second backward:', w.grad)

print('原因：同一张图重复 backward，w.grad 会继续 += 新贡献。')

## 3. Bug 2：忘记 zero_grad

下面用一个标量训练任务：让 `w` 靠近 `1`。

$$
loss = (w - 1)^2
$$

In [ ]:
def train_scalar(reset_grad_each_step):
    w = Value(5.0)
    history = []
    for step in range(5):
        loss = (w - 1) ** 2
        history.append((w.data, loss.data, w.grad))

        if reset_grad_each_step:
            w.grad = 0
        loss.backward()
        w.data += -0.1 * w.grad
    return history, w.data

for reset in [True, False]:
    history, final_w = train_scalar(reset)
    print()
    print('reset_grad_each_step =', reset)
    for step, (w_data, loss_data, old_grad) in enumerate(history):
        print(f'step {step}: w={w_data:.4f}, loss={loss_data:.4f}, grad_before_backward={old_grad:.4f}')
    print('final w:', round(final_w, 4))

## 4. Bug 3：learning_rate 太大

还是：

$$
loss = w^2
$$

如果学习率太大，参数可能越过最低点，甚至越来越远。

In [ ]:
def train_with_lr(lr, steps=6):
    w = Value(5.0)
    values = []
    for _ in range(steps):
        w.grad = 0
        loss = w ** 2
        loss.backward()
        values.append((w.data, loss.data, w.grad))
        w.data += -lr * w.grad
    return values

for lr in [0.1, 0.6, 1.1]:
    print()
    print('lr =', lr)
    for w_data, loss_data, grad in train_with_lr(lr):
        print(f'w={w_data:>8.4f}, loss={loss_data:>8.4f}, grad={grad:>8.4f}')

## 5. Bug 4：ReLU 断掉

ReLU 在负数区域导数是 0：

```text
x < 0 -> relu(x) = 0 -> grad = 0
```

所以如果某个神经元一直在负半轴，它这次不会把梯度传回去。

In [ ]:
for value in [-2.0, 3.0]:
    x = Value(value)
    y = x.relu()
    y.backward()
    print(f'x={value:>4}, relu={y.data:>4}, x.grad={x.grad}')

## 6. TODO Lab - 写 reset_grads

TODO：补全一个清空梯度的小函数。

In [ ]:
def reset_grads(parameters):
    # TODO: 把每个 p.grad 设为 0
    pass


def qa_check_09_reset_grads(func):
    params = [Value(1.0), Value(2.0), Value(3.0)]
    for i, p in enumerate(params):
        p.grad = i + 1
    out = func(params)
    if any(p.grad != 0 for p in params):
        print('请先填写 TODO：每个参数的 grad 都应该清零。')
        return
    print('OK: reset_grads passed.')


qa_check_09_reset_grads(reset_grads)

<details>
<summary><strong>Show / Hide 提示</strong></summary>

遍历 `parameters`，对每个 `p` 做：

```python
p.grad = 0
```

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def reset_grads(parameters):
    for p in parameters:
        p.grad = 0
```

</details>

## What To Remember

```text
1. grad 会累加，所以每轮 backward 前要清零。
2. 同一张计算图重复 backward，grad 可能变成两倍、三倍。
3. learning_rate 太小下降慢，太大可能震荡或变差。
4. ReLU 在负数区域不传梯度。
5. 调试顺序先看：loss、grad、zero_grad、update、learning_rate。
```

<!-- xiao-post:start -->
## 课后作业提交清单

```text
1. 我能解释为什么重复 backward 会让 grad 变大。
2. 我能解释为什么每轮训练要 zero_grad。
3. 我跑过不同 learning_rate，并看到太大时不稳定。
4. 我能解释 ReLU 负半轴为什么 grad 为 0。
5. 我能写出 reset_grads(parameters)。
```

## AI 复盘检查 Prompt

```text
你是我的 micrograd 梯度调试检查官。
请你一次只问一个问题，检查我是否理解：
1. 为什么重复 backward 会累加 grad
2. 为什么训练循环每轮都要 zero_grad
3. learning_rate 太大/太小分别会怎样
4. ReLU 为什么可能让 grad 变成 0
5. 如果 loss 不下降，我应该按什么顺序排查
如果我答错，请用 w=3, L=w^2 或 x=-2, relu(x) 的例子提示我。
```